## TF-IDF (Term Frequency - Inverse Document Frequency)
In Bag of Words every word gets the same importance. But words like *"the"*, *"was"*, *"and"* appear in almost every sentence, so they don't tell us much.

TF-IDF gives each word a **score** instead of a count:
- **TF (Term Frequency):** how often the word appears **in this sentence** → more times = higher score
- **IDF (Inverse Document Frequency):** in how many sentences the word appears **in the whole data** → appears everywhere = lower score, appears rarely = higher score

$$\text{TF} = \frac{\text{No. of times the word repeats in the sentence}}{\text{No. of words in the sentence}}$$

$$\text{IDF} = \ln\left(\frac{\text{No. of sentences}}{\text{No. of sentences containing the word}}\right)$$

$$\text{TF-IDF} = \text{TF} \times \text{IDF}$$

So a word gets a high score when it appears a lot in **this** sentence but is **rare** in the other sentences. That is exactly the kind of word that tells us what the sentence is about.

### Real life use case
**Search engines:** you search *"best biryani in Hyderabad"*. Words like *in* and *best* are on millions of pages, but *biryani* and *Hyderabad* are rare. TF-IDF gives these rare words more weight, so pages that are really about biryani in Hyderabad come on top.

**Spam filter:** *"call"* comes in both normal and spam messages, but words like *"prize"*, *"claim"* are rare and mostly in spam. TF-IDF gives them a higher score, so the model notices them more.

### Small example first
Same reviews as the Bag of Words notebook, so we can compare.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

reviews = [
    "the pizza was hot and tasty",
    "the pizza was cold and the delivery was late",
    "tasty food and fast delivery",
]

tfidf = TfidfVectorizer()
X = tfidf.fit_transform(reviews).toarray()

# round(2) only to make the table easy to read
pd.DataFrame(X, columns=tfidf.get_feature_names_out()).round(2)

,and,cold,delivery,fast,food,hot,late,pizza,tasty,the,was
0,0.31,0.00,0.00,0.00,0.00,0.52,0.00,0.40,0.40,0.40,0.40
1,0.21,0.35,0.27,0.00,0.00,0.00,0.35,0.27,0.00,0.53,0.53
2,0.32,0.00,0.41,0.53,0.53,0.00,0.00,0.00,0.41,0.00,0.00


Notice in the 1st sentence:
- **and** is in all 3 sentences → lowest score (0.31)
- **hot** is only in this sentence → highest score (0.52)

In Bag of Words both of them were just **1**. TF-IDF tells the model that *hot* is more important than *and*.

In [2]:
# IDF of every word: common words get a low IDF, rare words get a high IDF
# 'and' is in every sentence, so it has the smallest IDF
pd.Series(tfidf.idf_, index=tfidf.get_feature_names_out()).round(2).sort_values()

and         1.00
delivery    1.29
pizza       1.29
tasty       1.29
the         1.29
was         1.29
fast        1.69
cold        1.69
late        1.69
hot         1.69
food        1.69
dtype: float64

### Why is 'and' 0.31 and not 0?
With **our formula**, a word that is in every sentence gets IDF = 0:

$$\text{IDF of 'and'} = \ln\left(\frac{3}{3}\right) = \ln(1) = 0 \quad \Rightarrow \quad \text{TF-IDF} = 0$$

But sklearn changes the formula a little (default `smooth_idf=True`). It adds 1 to both numbers inside ln, and **adds 1 at the end**:

$$\text{IDF of 'and'} = \ln\left(\frac{1 + 3}{1 + 3}\right) + 1 = 0 + 1 = 1$$

That **+1** is on purpose: a word that is everywhere gets a **low** weight, but it is not thrown away completely. That's why the IDF table above shows **and = 1.00** (the lowest), not 0.

sklearn also changes TF: it uses only the **count** (No. of times the word repeats), it does **not divide** by the number of words in the sentence.

### How 0.31 is calculated (sentence 1: "the pizza was hot and tasty")
1. **TF:** every word comes once, so TF = 1
2. **TF × IDF:** and = 1 × 1.00 = 1.00, hot = 1 × 1.69 = 1.69, the / pizza / was / tasty = 1 × 1.29 = 1.29
3. **Scale the row** (`norm='l2'`): divide every value by the row's length √(1.00² + 1.69² + 4 × 1.29²) = √10.5 ≈ **3.24**
   - and = 1.00 / 3.24 = **0.31**
   - hot = 1.69 / 3.24 = **0.52**

So 'and' still gets the **lowest score in the sentence**, just not exactly 0.

Note: even `smooth_idf=False` keeps the +1 at the end, so sklearn never gives 0. The numbers are different, but the idea is the same in both: **common word → low score, rare word → high score**.

In [3]:
import numpy as np

total_sentences = 3

# in how many sentences each word appears (and -> 3, hot -> 1, ...)
sentences_with_word = (tfidf.transform(reviews).toarray() > 0).sum(axis=0)

pd.DataFrame({
    "sentences containing the word": sentences_with_word,
    "IDF (our formula)": np.log(total_sentences / sentences_with_word),
    "IDF (sklearn)": np.log((1 + total_sentences) / (1 + sentences_with_word)) + 1,
}, index=tfidf.get_feature_names_out()).round(2).sort_values("sentences containing the word", ascending=False)

,sentences containing the word,IDF (our formula),IDF (sklearn)
and,3,0.00,1.00
delivery,2,0.41,1.29
was,2,0.41,1.29
tasty,2,0.41,1.29
the,2,0.41,1.29
pizza,2,0.41,1.29
cold,1,1.10,1.69
late,1,1.10,1.69
hot,1,1.10,1.69
food,1,1.10,1.69


In [4]:
# TF-IDF of sentence 1 with our formula
words = "the pizza was hot and tasty".split()
idf_ours = pd.Series(np.log(total_sentences / sentences_with_word), index=tfidf.get_feature_names_out())

rows = []
for word in words:
    tf = words.count(word) / len(words)   # 1 / 6 for every word here
    idf = idf_ours[word]
    rows.append([word, tf, idf, tf * idf])

# 'and' -> 0 with our formula
pd.DataFrame(rows, columns=["word", "TF", "IDF", "TF-IDF"]).round(2)

,word,TF,IDF,TF-IDF
0,the,0.17,0.41,0.07
1,pizza,0.17,0.41,0.07
2,was,0.17,0.41,0.07
3,hot,0.17,1.10,0.18
4,and,0.17,0.00,0.00
5,tasty,0.17,0.41,0.07


In [5]:
# check sklearn's 0.31 by hand for sentence 1
idf_sklearn = pd.Series(tfidf.idf_, index=tfidf.get_feature_names_out())

scores = idf_sklearn[words]                         # TF = 1 for every word, so TF x IDF = IDF
length = np.sqrt((scores ** 2).sum())       # row length
print("length :", round(length, 2))
print("and    :", round(scores['and'] / length, 2))
print("hot    :", round(scores['hot'] / length, 2))

length : 3.24
and    : 0.31
hot    : 0.52


## Practical: SMS Spam Classification data
Same dataset as the Bag of Words practical: **5572 SMS messages**, each labelled as **ham** (normal) or **spam**.

In [7]:
messages = pd.read_csv('SMSSpamCollection.txt', sep='	', names=["label", "message"])
messages

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


### Data cleaning and preprocessing
Same steps as before, but this time we use **lemmatization** instead of stemming, so we get real words (*entry* instead of *entri*).

In [8]:
import re
import nltk

# stopwords -> list of stopwords
# wordnet   -> needed for the lemmatizer
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to C:\Users\Code and
[nltk_data]     Debug\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Code and
[nltk_data]     Debug\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [9]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

# set makes checking 'word in stop_words' much faster
stop_words = set(stopwords.words('english'))

In [11]:
corpus = []
for message in messages['message']:
    # [^a-zA-Z] means "anything that is NOT a letter" -> replace it with a space
    review = re.sub('[^a-zA-Z]', ' ', message)
    review = review.lower()
    words = review.split()

    clean_words = []
    for word in words:
        if word not in stop_words:
            lemma_word = lemmatizer.lemmatize(word)
            clean_words.append(lemma_word)

    # join the words back into one sentence
    corpus.append(' '.join(clean_words))

In [12]:
# compare one message before and after cleaning
print("Before:", messages['message'][2])
print("After :", corpus[2])

Before: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
After : free entry wkly comp win fa cup final tkts st may text fa receive entry question std txt rate c apply


In [ ]:
# corpus[:10]

### Create TF-IDF
`max_features=100` → keep only the **100 most frequent words** as columns.

In [ ]:
tfidf = TfidfVectorizer(max_features=100)
X = tfidf.fit_transform(corpus).toarray()

# 5572 messages (rows) and 100 words (columns)
X.shape

(5572, 100)

In [14]:
# the 100 words that were selected
tfidf.get_feature_names_out()

array(['already', 'also', 'amp', 'anything', 'ask', 'babe', 'back',
       'call', 'care', 'cash', 'claim', 'co', 'com', 'come', 'da', 'day',
       'dear', 'dont', 'find', 'free', 'friend', 'get', 'give', 'go',
       'going', 'good', 'got', 'great', 'gt', 'happy', 'hey', 'hi',
       'home', 'hope', 'im', 'know', 'last', 'later', 'let', 'life',
       'like', 'lor', 'love', 'lt', 'make', 'meet', 'message', 'min',
       'miss', 'mobile', 'morning', 'msg', 'much', 'need', 'new', 'night',
       'number', 'oh', 'ok', 'one', 'phone', 'please', 'pls', 'prize',
       'really', 'reply', 'right', 'said', 'say', 'see', 'send',
       'service', 'sorry', 'still', 'stop', 'take', 'tell', 'text',
       'thanks', 'thing', 'think', 'time', 'today', 'tomorrow', 'tone',
       'txt', 'uk', 'ur', 'want', 'wat', 'way', 'week', 'well', 'win',
       'work', 'would', 'www', 'yeah', 'year', 'yes'], dtype=object)

In [15]:
# view it as a table (pandas shows only some of the 100 columns)
# values are now decimal scores, not 0/1 like Bag of Words
pd.DataFrame(X, columns=tfidf.get_feature_names_out()).round(2).head()

,already,also,amp,anything,ask,babe,back,call,care,cash,...,way,week,well,win,work,would,www,yeah,year,yes
0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
1,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
2,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.57,0.0,0.0,0.0,0.0,0.0,0.0
3,0.46,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
4,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0


### `tfidf.vocabulary_`
Just like `cv.vocabulary_` in Bag of Words, it is a dictionary of **word → column number**. It tells us which column of `X` belongs to which word.

Careful: the number is **not the TF-IDF score**, it is only the column position (columns are in alphabetical order).

In [ ]:
# # np.int64(19) is just the number 19 (numpy's way of showing it)
# tfidf.vocabulary_

In [ ]:
# # which column is 'free'?
# col = tfidf.vocabulary_['free']
# col

In [ ]:
# # we pick message number 2 to check
# print(messages['message'][2])

# # it has the word 'free', so row 2 has a score in the 'free' column (0 if the word was not there)
# X[2, col]

In [16]:
# all the words of message 2 that have a score, highest first
# only 4 words: 'entry', 'cup', 'final' etc. are not in the top 100 words, so they have no column
row = pd.Series(X[2], index=tfidf.get_feature_names_out())
row[row > 0].sort_values(ascending=False)

win     0.571054
txt     0.491925
text    0.472951
free    0.456317
dtype: float64

### N-Grams
Same as Bag of Words: with `ngram_range=(2, 2)` every column is a **pair of words** (bigram) like *"free entry"* or *"call later"*, which keeps more meaning than single words.

In [ ]:
# tfidf = TfidfVectorizer(max_features=100, ngram_range=(2, 2))
# X = tfidf.fit_transform(corpus).toarray()

# X.shape

In [ ]:
# # now the columns are 2 word phrases
# tfidf.get_feature_names_out()

In [ ]:
# X